In [1]:
import requests
import csv
import sys

def fetch_and_save_popular_books():
    # 1. API 기본 정보 설정
    url = "http://data4library.kr/api/loanItemSrch"
    auth_key = '39c94fffc9e3a1975b677a0eccfa75178d693d534850006839f09ee79282bcf3'  # <-- 이곳에 실제 발급받은 인증키를 입력하세요.

    if auth_key == "YOUR_AUTH_KEY":
        print("오류: 인증키가 입력되지 않았습니다. 코드 내에 발급받은 authKey를 입력해주세요.")
        sys.exit()

    # 2. 파라미터 설정
    params = {
        "authKey": auth_key,
        "startDt": "2025-01-01",
        "endDt": "2026-06-30",
        "age": "20",
        "pageSize": 200,
        "format": "json"
    }

    try:
        print("도서관 정보나루 API에 데이터를 요청하는 중입니다...")
        # 3. API GET 요청
        response = requests.get(url, params=params)
        response.raise_for_status() # HTTP 오류 발생 시 예외 처리

        # 4. JSON 응답 데이터 파싱
        data = response.json()
        
        # 응답 구조 검증
        if 'response' in data and 'docs' in data['response']:
            books = data['response']['docs']
        else:
            print("API 응답에서 도서 데이터를 찾을 수 없습니다. 인증키나 파라미터를 다시 확인해주세요.")
            return

        # 5. CSV 파일로 저장
        filename = "popular_books_20s_2025_2026.csv"
        
        # Windows 엑셀 환경에서 한글 깨짐을 방지하기 위해 encoding='utf-8-sig' 사용
        with open(filename, mode='w', encoding='utf-8-sig', newline='') as file:
            # CSV 파일에 기록할 컬럼명(항목) 정의
            fieldnames = [
                'ranking', 'bookname', 'authors', 'publisher', 
                'publication_year', 'isbn13', 'class_nm', 'loan_count'
            ]
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            
            # 사용자 친화적인 한글 헤더 작성
            writer.writerow({
                'ranking': '순위',
                'bookname': '도서명',
                'authors': '저자명',
                'publisher': '출판사',
                'publication_year': '출판년도',
                'isbn13': 'ISBN13',
                'class_nm': '주제분류명',
                'loan_count': '대출건수'
            })

            # 수집된 도서 목록을 순회하며 행 추가
            for item in books:
                book = item['doc'] # 데이터 구조 상 doc 키 안에 정보가 들어있음
                writer.writerow({
                    'ranking': book.get('ranking', ''),
                    'bookname': book.get('bookname', ''),
                    'authors': book.get('authors', ''),
                    'publisher': book.get('publisher', ''),
                    'publication_year': book.get('publication_year', ''),
                    'isbn13': book.get('isbn13', ''),
                    'class_nm': book.get('class_nm', ''),
                    'loan_count': book.get('loan_count', '')
                })
                
        print(f"성공적으로 {len(books)}권의 인기 도서 데이터를 '{filename}' 파일로 저장했습니다.")
        
    except requests.exceptions.RequestException as e:
        print(f"API 네트워크 요청 중 오류가 발생했습니다: {e}")
    except Exception as e:
        print(f"데이터 처리 중 알 수 없는 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    fetch_and_save_popular_books()

도서관 정보나루 API에 데이터를 요청하는 중입니다...
성공적으로 200권의 인기 도서 데이터를 'popular_books_20s_2025_2026.csv' 파일로 저장했습니다.
